# Week 8: An introduction to pandas

AKA No more loops!

**pandas** is a Python library made for working with tables. 

### Questions:
- Where were the Giller nominees born, and where do they live now?
- How old are authors when they are first nominated?



## 1. Loading the data

`import pandas as pd` loads the library. This is a convention that most people use to abbreviate pandas, so you will see `pd.` at the start of many pandas commands.


In [ ]:
import pandas as pd

authors = pd.read_csv("giller_authors.csv")

print(len(authors), "authors")
type(authors)

In [ ]:
authors.head()

This looks a lot like a spreadsheet:
- The **column names** come from the header row of the CSV file.
- The numbers on the left (0, 1, 2, ...) are the **index**: pandas numbers the rows for us, starting from 0, just like a list.

## 2. Looking at one column

To get one column, we put its name in square brackets. This is just like getting a value from a dictionary, as in `book["Born"]`, except that we get the **whole column**.

A single column is called a **Series**.

In [ ]:
authors["Born region"]

## 3. Counting

Remember the counting loop we've written so many times?

```python
counts = {}
for author in birthplaces:
    region = ...
    if region in counts:
        counts[region] = counts[region] + 1
    else:
        counts[region] = 1
```



In pandas, that whole loop is one method: `.value_counts()`

In [ ]:
authors["Born in Canada"].value_counts()

Now let's count the regions where the nominees were born:

In [ ]:
born_counts = authors["Born region"].value_counts()

born_counts

The result of `.value_counts()` is itself a Series, so we can save it in a variable, `born_counts`, and keep working with it. For example, `.head(10)` gives us just the first 10 values: the 10 most common regions.

In [ ]:
born_counts.head(10)

## 4. Visualizing the data

(We will not test you on any plot making code.)

In [ ]:
import matplotlib.pyplot as plt

born_counts.head(10).plot.barh()
plt.show()

In [ ]:
ax = born_counts.head(10).plot.barh(color="#2a78d6")

ax.invert_yaxis()   # put the biggest bar at the top
ax.set_title("Giller nominees by region of birth (top 10 regions)")
ax.set_xlabel("Number of authors")
ax.set_ylabel("")   # the region names speak for themselves
plt.show()

## 5. The same question for residences

Now we can ask where the nominees live, by changing just the column name. 

In [ ]:
lives_counts = authors["Lives region"].value_counts()

ax = lives_counts.head(10).plot.barh(color="#eb6834")

ax.invert_yaxis()
ax.set_title("Giller nominees by region of current residence (top 10)")
ax.set_xlabel("Number of authors")
ax.set_ylabel("")
plt.show()

In [ ]:
ax.set_title("Giller nominees by region of current residence (top 10)")
ax.set_xlabel("Number of authors")
ax.set_ylabel("")
plt.show()

In [ ]:
# make a dictionary from the two Series
location_dict = {"Born": born_counts, "Lives": lives_counts}

# create a new dataframe, using fillna(0) to see the number of people
# to 0 where the data didn't exist in that series.
both = pd.DataFrame(location_dict).fillna(0)

# Sum the number of people born in a region with the number of people
# living in the region
both["Total"] = both["Born"] + both["Lives"]

# Keep the 10 regions with the most births plus residences combined
top = both.sort_values("Total", ascending=False).head(10)



In [ ]:
ax = top[["Born", "Lives"]].plot.barh(color=["#2a78d6", "#eb6834"])

ax.invert_yaxis()
ax.set_title("Giller nominees: region of birth vs. current residence")
ax.set_xlabel("Number of authors")
ax.set_ylabel("")
plt.show()

## 6. Choosing rows: filtering

In pandas, selecting rows is called **filtering**, and it takes two steps.

**Step 1:** Write a condition about a column. Just as `==` compares two values, here it compares *every* value in the column, and gives back a column of `True` and `False`.

In [ ]:
born_elsewhere = authors["Born in Canada"] == "Elsewhere"

born_elsewhere

**Step 2:** Now `born_elsewhere` is our boolean selector, and when we apply it to the `authors` dataframe, we get a new DataFrame with only the rows in which authors were born somewhere other than Canada.

In [ ]:
non_canadians = authors[born_elsewhere]

print(len(non_canadians), "authors born outside Canada")
non_canadians.head()

`non_canadians` is a new DataFrame with the same columns as `authors`, but only 55 rows. Everything we've learned works on it too. Where were these nominees born?

In [ ]:
# Step by step
count_of_non_canadians = non_canadians["Born region"].value_counts()

top_20 = count_of_non_canadians.head(20)

top_20

In [ ]:
# Now plot it
ax = top_20.plot.barh(color="#2a78d6")

ax.invert_yaxis()
ax.set_title("Nominees born outside Canada")
ax.set_xlabel("Number of authors")
ax.set_ylabel("")
plt.show()

But there are more regions with only one nominee. It is just an accident that these ones are represented. Should we leave them out? 

In [ ]:
# Let's split up these operations and plot all regions (countries) that
# have at least 2 nominees.

non_can_counts = non_canadians["Born region"].value_counts()

# Then create a boolean selector for counts that are at least two
non_can_at_least_two = non_can_counts >= 2

# We can finally get the Series that we want to plot.
plotdata = non_can_counts[non_can_at_least_two]

print(plotdata)

In [ ]:
ax = plotdata.plot.barh(color="#2a78d6")

ax.invert_yaxis()
ax.set_title("Nominees born outside Canada")
ax.set_xlabel("Number of authors")
ax.set_ylabel("")
plt.show()

## 7. From counts to percentages

Percentages often make it easier to do comparisons.

`.value_counts(normalize=True)` gives each value's *fraction* of the rows, instead of its count:

In [ ]:
authors["Born in Canada"].value_counts(normalize=True)

These fractions add up to 1. To turn them into percentages, we multiply by 100. Notice that we don't need a loop: pandas does the calculation for every value in the Series. Then `.round(1)` rounds every value to one decimal place.

In [ ]:
born_percent = authors["Born in Canada"].value_counts(normalize=True) * 100
born_percent = born_percent.round(1)

born_percent

### Looking up one value

The result is a Series, and each value has a label on the left: `Canada`, `Elsewhere`, and `Unknown`. To get one value, we put its label in square brackets, just as we look up a key in a dictionary. This works on the result of a plain `.value_counts()` too.

Let's use a lookup to compare where the nominees were born with where they live now.

In [ ]:
lives_percent = authors["Lives in Canada"].value_counts(normalize=True) * 100
lives_percent = lives_percent.round(1)

print("Born in Canada:", born_percent["Canada"], "%")
print("Living in Canada:", lives_percent["Canada"], "%")

## 8. How old are authors when they are first nominated?

Is the Giller a prize for established writers late in their careers, or does it find new voices? One way to approach this question is to ask **how old authors were when they were first nominated**.

In [ ]:
books = pd.read_csv("giller_nominees_clean.csv")

# Get the columns we care about
important_columns = ["Status", "Year", "Title", "Author", "DOB"]
books = books[important_columns]

print(len(books), "nominated books")
books.head()

### Looking at the dates of birth

The dates in the `DOB` column are not all written the same way:
- a full date, like `09/06/1959` (day/month/year), sometimes without the zeros, like `4/4/1943`;
- just a month and year, like `10/1969`;
- just a year, like `1963`;
- a `-` when the date of birth is unknown.

Before we use a column, it's always worth checking for surprises. Here are two authors whose dates are written in different ways. `.isin()` keeps the rows whose author is in our list:

In [ ]:
books[books["Author"].isin(["Paulette Jiles", "Emma Donoghue"])]


### Keeping just the year

We've used slices to take part of a list or a string. A negative number counts from the end, so `[-4:]` means "from the fourth-last character to the end":

In [ ]:
dob = "09/06/1959"
print(dob[-4:])

To do that to every value in a column, we write `.str` first, just as we do with `.str.split()`. We save the result as a new column, `Birth year`.

In [ ]:
books["Birth year"] = books["DOB"].str[-4:]

books[books["Author"].isin(["Paulette Jiles", "Emma Donoghue"])]

## 9. Calculating the ages

Some birth years are unknown, so we need to do three things:

1. **Leave out the unknown birth years.** 
2. **Turn the birth years into numbers.** T
3. **Subtract.** Subtracting one column from another subtracts the values row by row, so one line calculates the age for every book.

In [ ]:
no_birth_year = books["Birth year"] != "-"
known_books = books[no_birth_year].copy()

# convert years to integer (from text)
known_books["Birth year"] = known_books["Birth year"].astype(int)

known_books["Age"] = known_books["Year"] - known_books["Birth year"]

print(len(books) - len(known_books), "books left out because the date of birth is unknown")
known_books[["Year", "Title", "Author", "Age"]].head()

## 10. Each author's first nomination

Many authors were nominated more than once, but we want the age at their **first** nomination. We can find it in two steps:

1. Sort the books by year 
2. Keep the first row for each author

In [ ]:
books_by_year = known_books.sort_values("Year")

# drop_duplicates will keep the first one
first_nominations = books_by_year.drop_duplicates("Author")

print(len(first_nominations), "authors")
print(len(authors) - len(first_nominations), "authors left out because their date of birth is unknown")

Let's check that it worked for an author with several nominations. Emma Donoghue was nominated in 2008, 2016, and 2020:

In [ ]:
first_nominations[first_nominations["Author"] == "Emma Donoghue"]

Who were the youngest and oldest first-time nominees? Sorting by `Age` puts the youngest first. `.head()` shows the first rows (the youngest), and `.tail()` shows the last rows (the oldest).

In [ ]:
by_age = first_nominations.sort_values("Age")

by_age[["Author", "Year", "Age"]].head()

In [ ]:
by_age[["Author", "Year", "Age"]].tail()

Now we can look at the median and mean (without having to know how to compute them)

In [ ]:
median_age = first_nominations["Age"].median()
mean_age = first_nominations["Age"].mean()

print("Median age at first nomination:", median_age)
print("Average age at first nomination:", round(mean_age, 1))

## 11. A chart of ages: the histogram

For a column of numbers like ages, the natural chart is a **histogram**. It divides the ages into ranges, called **bins** (here, 5 years wide: 25–29, 30–34, and so on), and draws one bar for each bin showing how many authors fall into it.

`bins=range(25, 90, 5)` sets where the bins start: 25, 30, 35, and so on. `ax.axvline()` draws a vertical line, which we use to mark the median.

In [ ]:
ax = first_nominations["Age"].plot.hist(bins=range(25, 90, 5), color="#2a78d6", edgecolor="white")

ax.axvline(median_age, color="black", linewidth=1)   # mark the median

ax.set_title("The typical author is first nominated for the Giller in their 40s")
ax.set_xlabel("Age at first Giller nomination")
ax.set_ylabel("Number of authors")
plt.show()


## 12. How old are authors when they win?

Because we calculated an age for every book, comparing winners takes only a filter. The `Status` column marks winning books as `winner` or `winner (tie)`, and we'll include both. Each winning book is one row, so an author who wins more than once appears more than once.

In [ ]:
winners = known_books[known_books["Status"].isin(["winner", "winner (tie)"])]

print(len(winners), "winning books")
print("Median age when winning:", winners["Age"].median())
print("Average age when winning:", round(winners["Age"].mean(), 1))